# Notatnik 1 ? Klasyfikacja Iris z Random Forest

Zbudujemy pierwszy model klasyfikacyjny. Model dostanie cztery pomiary kwiatu i spr?buje przewidzie? gatunek irysa.

Najwa?niejszy cel nie brzmi ?dosta? wysok? liczb??, tylko zrozumie? ca?y przep?yw:

```text
dane ? X/y ? train/test ? model.fit() ? model.predict() ? metryki ? interpretacja b??d?w
```

## 1. Import i wczytanie danych

Dataset Iris jest ma?y, czysty i dost?pny bez pobierania plik?w. To dobry pierwszy przyk?ad, bo od razu mo?emy skupi? si? na przep?ywie ML.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris(as_frame=True)
df = iris.frame

print("Dataset Iris za?adowany.")
print("Liczba wierszy:", len(df))
print("Liczba kolumn:", len(df.columns))

df.head()

## 2. Co oznaczaj? kolumny?

Cechy wej?ciowe to pomiary kwiatu, a `target` to poprawna odpowied?. Model nie ?widzi? nazwy gatunku jako tekstu, tylko numer klasy.

In [ ]:
print("Cechy:")
for feature in iris.feature_names:
    print("-", feature)

print("\nMapowanie target -> nazwa gatunku:")
for number, name in enumerate(iris.target_names):
    print(f"{number} -> {name}")

## 3. Oddzielamy `X` i `y`

- `X` to dane wej?ciowe, czyli cechy.
- `y` to odpowied?, kt?rej model ma si? nauczy?.

In [ ]:
X = iris.data
y = iris.target

print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Podzia? na trening i test

U?ywamy `stratify=y`, ?eby w zbiorze treningowym i testowym zachowa? podobne proporcje klas. To dobra praktyka w klasyfikacji, szczeg?lnie gdy klasy nie s? idealnie r?wne.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Proporcje klas w train:")
print(y_train.value_counts(normalize=True).sort_index())
print("\nProporcje klas w test:")
print(y_test.value_counts(normalize=True).sort_index())

## 5. Baseline: najprostszy punkt odniesienia

Zanim ucieszymy si? z wyniku modelu, sprawdzamy naiwny baseline. `DummyClassifier` przewiduje najcz?stsz? klas?. Dobry model musi by? wyra?nie lepszy od takiego punktu odniesienia.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy: {baseline_accuracy:.2%}")

## 6. Trenujemy Random Forest

Random Forest to zesp?? wielu drzew decyzyjnych. Pojedyncze drzewo ?atwo przeucza si? do danych, a las u?rednia decyzje wielu drzew, co zwykle daje stabilniejszy model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)

model.fit(X_train, y_train)
print("Model wytrenowany.")

## 7. Ocena modelu

Accuracy jest intuicyjne, ale nie m?wi, kt?re klasy model myli. Dlatego dodamy macierz pomy?ek i raport klasyfikacji.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Random Forest accuracy: {accuracy:.2%}")
print(f"Poprawa wzgl?dem baseline: {accuracy - baseline_accuracy:.2%}")

print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=iris.target_names))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=iris.target_names,
    cmap="Blues",
)
plt.title("Macierz pomy?ek ? Iris")
plt.show()

## 8. Kt?re cechy by?y wa?ne?

Random Forest potrafi oszacowa?, kt?re cechy najcz??ciej pomaga?y w podziale danych. To nie jest pe?ne wyja?nienie modelu, ale dobry pierwszy sygna?.

In [ ]:
importance = pd.Series(model.feature_importances_, index=iris.feature_names).sort_values()

importance.plot(kind="barh", title="Wa?no?? cech wed?ug Random Forest")
plt.xlabel("feature_importance")
plt.show()

importance.sort_values(ascending=False)

## 9. Predykcja dla nowego kwiatu

Podajemy dane w tej samej kolejno?ci kolumn, jak przy trenowaniu modelu.

In [ ]:
nowy_kwiat = pd.DataFrame(
    [[6.7, 3.1, 4.7, 1.5]],
    columns=iris.feature_names,
)

prediction = model.predict(nowy_kwiat)[0]
probabilities = model.predict_proba(nowy_kwiat)[0]

print("Przewidziany gatunek:", iris.target_names[prediction])
print("\nPrawdopodobie?stwa klas:")
for name, probability in zip(iris.target_names, probabilities):
    print(f"{name}: {probability:.2%}")

## Twoja kolej

1. Zmie? warto?ci w `moj_kwiat`.
2. Sprawd? predykcj? i prawdopodobie?stwa.
3. Zastan?w si?, czy wynik jest pewny, czy model si? waha.

In [ ]:
moj_kwiat = pd.DataFrame(
    [[5.9, 3.0, 5.1, 1.8]],
    columns=iris.feature_names,
)

moja_predykcja = model.predict(moj_kwiat)[0]
moje_prawdopodobienstwa = model.predict_proba(moj_kwiat)[0]

print("Przewidziany gatunek:", iris.target_names[moja_predykcja])
for name, probability in zip(iris.target_names, moje_prawdopodobienstwa):
    print(f"{name}: {probability:.2%}")

## Podsumowanie

Po tym notebooku umiesz:

- rozdzieli? dane na `X` i `y`,
- zrobi? sensowny `train_test_split` dla klasyfikacji,
- por?wna? model z baseline'em,
- oceni? klasyfikacj? przez accuracy, confusion matrix i classification report,
- wykona? predykcj? dla nowego przyk?adu,
- sprawdzi?, kt?re cechy by?y wa?ne dla Random Forest.